# EGCO461 Project: Thai Sentiment Classification (Baseline)

Notebook นี้เป็นจุดเริ่มต้น: โหลดข้อมูล ตัดคำภาษาไทย สร้าง Naïve Bayes baseline วัดผลด้วย **macro-F1** และสร้างไฟล์ส่งผล

หน้าที่ของกลุ่มคือทำให้ดีกว่า baseline นี้ และอธิบายได้ว่า **อะไรช่วยได้จริง เพราะอะไร**

In [ ]:
# !pip install pythainlp scikit-learn pandas  # run once if needed
import pandas as pd
from pythainlp.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score, classification_report, confusion_matrix

## 1. Load data

In [ ]:
train = pd.read_csv('data/train.csv')
valid = pd.read_csv('data/valid.csv')
sample = pd.read_csv('data/sample_labeled.csv')   # 100 labeled examples in the same style as the test set
test = pd.read_csv('data/test.csv')               # no labels
print(train.shape, valid.shape, sample.shape, test.shape)
train['category'].value_counts()

สังเกตว่าคลาสไม่สมดุลมาก (q มีน้อย) จึงใช้ macro-F1 ซึ่งให้น้ำหนักทุกคลาสเท่ากัน

In [ ]:
train.sample(10, random_state=0)

## 2. Tokenize Thai text

ภาษาไทยไม่มีช่องว่างระหว่างคำ ต้องตัดคำก่อน (ใช้ `newmm` ของ PyThaiNLP)

In [ ]:
def thai_tokenize(text):
    return word_tokenize(str(text), engine='newmm', keep_whitespace=False)

thai_tokenize('อร่อยมากกก ร้านนี้ต้องมาอีก 555')

## 3. Baseline: bag of words + Multinomial Naïve Bayes

In [ ]:
vectorizer = CountVectorizer(tokenizer=thai_tokenize, token_pattern=None)
X_train = vectorizer.fit_transform(train['text'])
X_valid = vectorizer.transform(valid['text'])
X_sample = vectorizer.transform(sample['text'])

model = MultinomialNB(alpha=1.0)   # Laplace smoothing, same idea as in the lecture
model.fit(X_train, train['category'])

## 4. Evaluate

In [ ]:
labels = ['pos', 'neu', 'neg', 'q']
for name, X, y in [('valid', X_valid, valid['category']), ('sample', X_sample, sample['category'])]:
    pred = model.predict(X)
    print(f"{name}: macro-F1 = {f1_score(y, pred, labels=labels, average='macro'):.3f}")

print(classification_report(valid['category'], model.predict(X_valid), labels=labels, digits=3, zero_division=0))
pd.DataFrame(confusion_matrix(valid['category'], model.predict(X_valid), labels=labels), index=labels, columns=labels)

**คำถามชวนคิด:** ทำไม recall ของ `pos` และ `q` ต่ำมาก? ลองดูตัวอย่างที่ทายผิดในเซลล์ถัดไป

In [ ]:
pred = model.predict(X_valid)
errors = valid.assign(pred=pred)[lambda d: d['category'] != d['pred']]
errors.sample(15, random_state=1)[['text', 'category', 'pred']]

## 5. Create a submission file

ไฟล์ต้องมีคอลัมน์ `id,category` ครบทุก id ใน `test.csv` ตั้งชื่อไฟล์ให้สื่อถึงโมเดล เช่น `G05_run1_nb.csv` (ส่งได้สูงสุด 5 ไฟล์)

In [ ]:
submission = pd.DataFrame({'id': test['id'], 'category': model.predict(vectorizer.transform(test['text']))})
submission.to_csv('G00_run1_nb_baseline.csv', index=False)
submission['category'].value_counts()

## Ideas to try (เลือกเองว่าจะลองอะไร แล้ววัดผลให้ชัด)

- TF-IDF, n-grams (`ngram_range=(1, 2)`), character n-grams
- Logistic Regression / Linear SVM, `class_weight='balanced'`
- จัดการข้อความ: ตัวอักษรซ้ำ (มากกกก), 555, อีโมจิ, ลิงก์
- ปรับ threshold หรือ resampling เพื่อช่วยคลาสเล็ก
- Pretrained Thai models เช่น WangchanBERTa (ต้องใช้ GPU เช่น Google Colab)
- **สำคัญ:** ใช้ `valid.csv` และ `sample_labeled.csv` เลือกโมเดล อย่าเดาจาก test